In [ ]:
!pip install gmplot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import radians, cos, sin, asin, sqrt
import gmplot as gmplot
import pandas as pd
import os

In [ ]:
def Draw_Plots(time, Position, Speed, Acceleration, filename, Time_Section, transparency):

    # Set subplots in a large figure
    fig, (subfig1, subfig2, subfig3) = plt.subplots(3, figsize = (9,8))

    # Plot Position
    subfig1.plot(time, Position)

    for i in range(Time_Section.shape[0]):
        subfig1.fill_betweenx([0,max(Position)], [Time_Section[i,0],Time_Section[i,0]], [Time_Section[i,1],Time_Section[i,1]],
                                color = [0.7, 0.7, 0.7], alpha = transparency)
    # subfig1.fill_betweenx([0,max(Position)], [Time_Section[2],Time_Section[2]], [Time_Section[3],Time_Section[3]],
    #                         color = [0.255, 0.255, 0], alpha = transparency)
    subfig1.set_ylabel('Position (m)')
    subfig1.grid(True)
    subfig1.minorticks_on()
    subfig1.grid(which='minor', linestyle = ':')
    subfig1.set_xlabel('Time (s)')

    # Plot Speed
    subfig2.plot(time, Speed)
    for i in range(Time_Section.shape[0]):
        subfig2.fill_betweenx([0,120], [Time_Section[i,0],Time_Section[i,0]], [Time_Section[i,1],Time_Section[i,1]],
                                color = [0.7, 0.7, 0.7], alpha = transparency)
    # subfig2.fill_betweenx([0,120], [Time_Section[2],Time_Section[2]], [Time_Section[3],Time_Section[3]],
    #                         color = [0.255, 0.255, 0], alpha = transparency)
    subfig2.set_ylabel('Speed (km/h)')
    subfig2.grid(True)
    subfig2.minorticks_on()
    subfig2.grid(which='minor', linestyle = ':')
    subfig2.set_xlabel('Time (s)')
    subfig2.set_ylim([0,120])

    # Plot Acceleration
    subfig3.plot(time, Acceleration)
    for i in range(Time_Section.shape[0]):
        subfig3.fill_betweenx([-4,3], [Time_Section[i,0],Time_Section[i,0]], [Time_Section[i,1],Time_Section[i,1]],
                                color = [0.7, 0.7, 0.7], alpha = transparency)
    # subfig3.fill_betweenx([-4,3], [Time_Section[2],Time_Section[2]], [Time_Section[3],Time_Section[3]],
    #                         color = [0.255, 0.255, 0], alpha = transparency)
    subfig3.set_ylabel('Acc (m/s^2)')
    subfig3.set_xlabel('Time (s)')
    subfig3.set_ylim([-4,3])
    subfig3.grid(True)
    subfig3.minorticks_on()
    subfig3.grid(which='minor', linestyle = ':')


    fig.suptitle('Vehicle States')
    plt.tight_layout()

    # Save figures
    print(filename)
    plt.savefig(filename + ' States.png')
    plt.show()

In [ ]:
def Accel_and_Pos(Speed, Latitude, Longitude, dt = 0.04):

    # Calculate acceleration in m/s^2
    Acceleration = np.gradient(np.multiply(Speed,(5/18)),dt)

    # Get Moving Average of Acceleration to filter the noise
    windowsize = 10
    for i in range(len(Acceleration)):
        Acceleration[i] = Acceleration[i:i+windowsize].mean()

    # Compute Position(m)
    Position = haversine(Latitude, Longitude)

    return Acceleration, Position

In [ ]:
def Route_Section(file, Start, End, Direction, Section):

    filename = directory + file
    # print(filename)

    file = pd.read_csv(filename)

    # Pull data from CSV file
    Latitude = file['Latitude'].tolist()
    Longitude = file['Longitude'].tolist()
    Speed = file['Speed'].tolist()
    Long_Time = file['Time'].tolist()

    Latitude = np.array(Latitude)
    Longitude = np.array(Longitude)
    Speed = np.array(Speed)
    Long_Time = np.array(Long_Time)

    # Convert Speed from m/s to km/h
    Speed = Speed * 3.6

    # Discretize time @25Hz to start from the first sliced value
    time = np.arange(0, len(Speed)*dt, dt)

    # Create a bolean index based on the Start - End GPS coordinates to slice data
    match Direction:

        case 'Lat':

            Bol_index = (Latitude >= Start) & (Latitude <= End)

        case 'Lon':

            Bol_index = (Longitude >= Start) & (Longitude <= End)

        case _:
            print('Invalid geografical direction, Choose Lat or Lon')

    # Slice data based on Bolean index
    Longitude = Longitude[Bol_index]
    time = time[Bol_index]
    Long_Time = Long_Time[Bol_index]
    Speed = Speed[Bol_index]
    Latitude = Latitude[Bol_index]

    # Calculate Acceleration and position
    Acceleration, Position = Accel_and_Pos(Speed, Latitude, Longitude)
    # Remove extra letters from time str
    Long_Time = Fix_Time(Long_Time)

    # Create data structure to save as a CSV file
    data = {
        'Time': Long_Time,
        'Latitude': Latitude,
        'Longitude': Longitude,
        'Position': Position,
        'Speed': Speed,
        'Acceleration': Acceleration
    }

    filename = filename.replace("RaceBox Track Sessionon 28-02-2026 ","Section " + str(Section+1) + " Route ")
    df = pd.DataFrame(data)
    df.to_csv(filename)

    filename = filename.replace(".csv", "")

    # Record time length of the Section
    Time_Section = np.array([[time[0],time[-1]]])

    # Create and save vehicle states and car route in Gmaps
    Draw_Plots(time, Position, Speed, Acceleration, filename, Time_Section, 0)
    Gmaps(Latitude, Longitude, filename)

    return Time_Section

In [ ]:
# Calculate Position(m) based on GPS coordinates, check haversine formula
def haversine(lat, lon):

    Position = np.zeros(len(lat))
    Position[0] = 0

    for i in range(len(lat)-1):
        lat1 = lat[i]
        lat2 = lat[i+1]
        lon1 = lon[i]
        lon2 = lon[i+1]

        # distance between latitudes
        # and longitudes
        dLat = (lat2 - lat1) * np.pi / 180.0
        dLon = (lon2 - lon1) * np.pi / 180.0

        # convert to radians
        lat1 = (lat1) * np.pi / 180.0
        lat2 = (lat2) * np.pi / 180.0

        # apply formulae
        a = (pow(np.sin(dLat / 2), 2) +
            pow(np.sin(dLon / 2), 2) *
                np.cos(lat1) * np.cos(lat2))

        R = 6378137; # Earth radius in METERS
        c = 2 * np.sin(np.sqrt(a))

        Position[i+1] = Position[i] + R * c

    return Position

In [ ]:
def Data_Processing(file):

    filename = directory + file
    print(filename)

    file = pd.read_csv(filename)

    Latitude = file['Latitude'].tolist()
    Longitude = file['Longitude'].tolist()
    #Altitude = file['Altitude'].tolist()
    Speed = file['Speed'].tolist()
    Speed = np.array(Speed)
    Speed = Speed * 3.6

    # Calculate Acceleration and position
    Acceleration, Position = Accel_and_Pos(Speed, Latitude, Longitude)
    print(min(Acceleration))

    time = np.arange(0, len(Speed)*dt, dt)

    filename = filename.replace("RaceBox 24-10-2025 ","")
    filename = filename.rstrip('.csv')

    Draw_Plots(time, Position, Speed, Acceleration, filename, Time_Section, 0.3)

In [ ]:
def Gmaps(Latitude, Longitude, filename):

  # Create Google maps object based on initial coordinates and zoom
  gmap3 = gmplot.GoogleMapPlotter(31.8188, -106.5156, 13)
  # gmap3 = gmplot.GoogleMapPlotter(31.777987, -106.442673, 13)   #Centered on Spagetti bowl

  # Plot method Draw a line in between given coordinates
  gmap3.plot(Latitude, Longitude, 'cornflowerblue', edge_width = 2.5)

  gmap3.draw( filename + ".html" )

  return

In [ ]:
def Fix_Time(Time):
  # Remove additional leters from Time str
    Time = [time_str.replace("T", " ").replace("Z", "") for time_str in Time]
    return Time

In [ ]:
def Match_Radar_Time(file,file_foll):

  # Racebox_file = pd.read_csv('/content/RaceBox Track Sessionon 28-02-2026 10-04.csv')
  # Radar_file = pd.read_csv('/content/Mesa_2_KF_Final.csv')

  Racebox_file = pd.read_csv(file)
  Radar_file = pd.read_csv(file_foll)

  Racebox_Time = Racebox_file['Time'].tolist()
  Radar_Time = Radar_file['timestamp'].tolist()
  Frame_index = Radar_file['frame_index_zeroed'].tolist()
  Frame_index = np.array(Frame_index)
  # Frame_index = np.delete(Frame_index, -1)

  if (Frame_index[0] != 0):
    #Ensure index starts from 0
    Frame_index = Frame_index - Frame_index[0]


  # Make data pd.datetime64
  # Add day to Radar data
  # date = "2026-02-28"
  Racebox_Time = pd.to_datetime(Racebox_Time)
  Radar_Time = pd.to_datetime(Radar_Time)
  # Radar_Time = pd.to_datetime([f"{date} {t}" for t in Radar_Time])

  # Align Time to UTC
  Racebox_Time = Racebox_Time.tz_localize('UTC')
  Radar_Time = Radar_Time.tz_localize('America/Ciudad_Juarez')
  Radar_Time = Radar_Time.tz_convert('UTC')

  # Add Average Time offset of +0.8 sec PLUS +20 sec delay from camera
  Radar_Time = Radar_Time + pd.Timedelta(seconds=20.8)

  # Create a mask to slice data
  start = pd.Timestamp(Radar_Time[0])
  end = pd.Timestamp(Radar_Time[-1])

  mask = (Racebox_Time >= start) & (Racebox_Time <= end)
  Racebox_Time = Racebox_Time[mask]

  start2 = pd.Timestamp(Racebox_Time[0])
  end2 = pd.Timestamp(Racebox_Time[-1])

  mask2 = (Radar_Time >= start2) & (Radar_Time <= end2)
  Radar_Time = Radar_Time[mask2]

  # Make both arrays same length
  Frame_index = Frame_index[mask2]
  Racebox_Time = Racebox_Time[Frame_index]

  elapsed_time = (Racebox_Time - Racebox_Time[0]).total_seconds()
  print(f'{elapsed_time = }')
  print(f'{Radar_Time = }')
  print(f'{Racebox_Time = }')
  print(f'{Frame_index = }')

  return mask, Frame_index, elapsed_time


#I-10 - EV Follower 1
Sync Rcebox and Radar

In [ ]:
directory = '/content/'
file = 'RaceBox Track Sessionon 28-02-2026 13-14.csv'
file_foll = 'I10_FirstVeh_Smoothed_June17.csv'

filename = directory + file

mask, index, elapsed_time = Match_Radar_Time(file,file_foll)

Racebox_file = pd.read_csv(file)

Latitude = Racebox_file['Latitude'].tolist()
Longitude = Racebox_file['Longitude'].tolist()
#Altitude = file['Altitude'].tolist()
Speed = Racebox_file['Speed'].tolist()
Long_Time = Racebox_file['Time'].tolist()
Speed = np.array(Speed)
Speed = Speed * 3.6

Latitude = np.array(Latitude)
Longitude = np.array(Longitude)
#Altitude = np.array(Altitude)
Speed = np.array(Speed)
Long_Time = np.array(Long_Time)

Latitude = Latitude[mask]
Longitude = Longitude[mask]
#Altitude = Altitude[mask]
Speed = Speed[mask]
Long_Time = Long_Time[mask]

Latitude = Latitude[index]
Longitude = Longitude[index]
#Altitude = Altitude[index]
Speed = Speed[index]
Long_Time = Long_Time[index]

# Calculate Acceleration and position
Acceleration, Position = Accel_and_Pos(Speed, Latitude, Longitude)

makefile = {
    'Time': Long_Time,
    'Latitude': Latitude,
    'Longitude': Longitude,
    'Position (m)': Position,
    'Speed (km/h)': Speed,
    'Acceleration (m/s2)': Acceleration
}

filename = filename.replace("RaceBox Track Sessionon 28-02-2026 ","I10-Sync_EV_Follower1_data ")
df = pd.DataFrame(makefile)
df.to_csv(filename)

filename = filename.replace(".csv", "")

# Record time length of the Section
Time_Section = np.array([[elapsed_time[0],elapsed_time[-1]]])

Draw_Plots(elapsed_time, Position, Speed, Acceleration, filename, Time_Section, 0)
Gmaps(Latitude, Longitude, filename)

#I-10 - EV Follower 2
Sync Rcebox and Radar

In [ ]:
directory = '/content/'
file = 'RaceBox Track Sessionon 28-02-2026 13-14.csv'
file_foll = 'I10_SecondVeh_Smoothed_June17.csv'

filename = directory + file

mask, index, elapsed_time = Match_Radar_Time(file,file_foll)

Racebox_file = pd.read_csv(file)

Latitude = Racebox_file['Latitude'].tolist()
Longitude = Racebox_file['Longitude'].tolist()
#Altitude = file['Altitude'].tolist()
Speed = Racebox_file['Speed'].tolist()
Long_Time = Racebox_file['Time'].tolist()
Speed = np.array(Speed)
Speed = Speed * 3.6

Long_Time = Fix_Time(Long_Time)

Latitude = np.array(Latitude)
Longitude = np.array(Longitude)
#Altitude = np.array(Altitude)
Speed = np.array(Speed)
Long_Time = np.array(Long_Time)

Latitude = Latitude[mask]
Longitude = Longitude[mask]
#Altitude = Altitude[mask]
Speed = Speed[mask]
Long_Time = Long_Time[mask]

Latitude = Latitude[index]
Longitude = Longitude[index]
#Altitude = Altitude[index]
Speed = Speed[index]
Long_Time = Long_Time[index]

# Calculate Acceleration and position
Acceleration, Position = Accel_and_Pos(Speed, Latitude, Longitude)

makefile = {
    'Time': Long_Time,
    'Latitude': Latitude,
    'Longitude': Longitude,
    'Position (m)': Position,
    'Speed (km/h)': Speed,
    'Acceleration (m/s2)': Acceleration
}

filename = filename.replace("RaceBox Track Sessionon 28-02-2026 ","I10-Sync_EV_Follower2_data ")
df = pd.DataFrame(makefile)
df.to_csv(filename)

filename = filename.replace(".csv", "")

# Record time length of the Section
Time_Section = np.array([[elapsed_time[0],elapsed_time[-1]]])

Draw_Plots(elapsed_time, Position, Speed, Acceleration, filename, Time_Section, 0)
Gmaps(Latitude, Longitude, filename)


#Mesa - EV Follower 1
Sync Rcebox and Radar

In [ ]:
directory = '/content/'
file = 'RaceBox Track Sessionon 22-03-2026 12-43.csv' # Racebox data file
file_foll = 'Mesa_8_KF_Final_Rep_Removed.csv'         # Processed Radar data file

filename = directory + file

mask, index, elapsed_time = Match_Radar_Time(file,file_foll)

Racebox_file = pd.read_csv(file)

Latitude = Racebox_file['Latitude'].tolist()
Longitude = Racebox_file['Longitude'].tolist()
#Altitude = file['Altitude'].tolist()
Speed = Racebox_file['Speed'].tolist()
Long_Time = Racebox_file['Time'].tolist()
Speed = np.array(Speed)
Speed = Speed

# Long_Time = Fix_Time(Long_Time)

Latitude = np.array(Latitude)
Longitude = np.array(Longitude)
#Altitude = np.array(Altitude)
Speed = np.array(Speed)
Long_Time = np.array(Long_Time)

Latitude = Latitude[mask]
Longitude = Longitude[mask]
#Altitude = Altitude[mask]
Speed = Speed[mask]
Long_Time = Long_Time[mask]

Latitude = Latitude[index]
Longitude = Longitude[index]
#Altitude = Altitude[index]
Speed = Speed[index]
Long_Time = Long_Time[index]

# Calculate Acceleration and position
Acceleration, Position = Accel_and_Pos(Speed, Latitude, Longitude)

makefile = {
    'Time': Long_Time,
    'Latitude': Latitude,
    'Longitude': Longitude,
    'Position (m)': Position,
    'Speed (km/h)': Speed,
    'Acceleration (m/s2)': Acceleration
}

filename = filename.replace("RaceBox Track Sessionon 22-03-2026 ","Mesa-Sync_ICE_Follower1_data ")
df = pd.DataFrame(makefile)
df.to_csv(filename)

filename = filename.replace(".csv", "")

# Record time length of the Section
Time_Section = np.array([[elapsed_time[0],elapsed_time[-1]]])

Draw_Plots(elapsed_time, Position, Speed, Acceleration, filename, Time_Section, 0)
Gmaps(Latitude, Longitude, filename)


In [ ]:
dt = 0.04
directory = '/content/'
file = 'RaceBox Track Sessionon 22-03-2026 12-43.csv'
Lat_or_Lon = 'Lat'

'''
Start_End = np.array([[-106.484523,-106.464403],
             [-106.461056,-106.423706],
             [-106.399429,-106.301123]])
'''
Start_End = np.array([[31.767485, 31.777729],   # Section 1 Start/End
                      [31.782560, 31.791913],   # Section 2 Start/End
                      [31.818124, 31.828806]])  # Section 3 Start/End

Time_Section = np.zeros(Start_End.shape)

for i in range(0,Start_End.shape[0]):
    Time_Section[i,:] = Route_Section(file, Start_End[i, 0], Start_End[i, 1], Lat_or_Lon, i)
    print(Time_Section)


Data_Processing(file)

# for entry in os.scandir(directory):
#     if entry.is_file():
#         file = str(entry)
#         file = file.lstrip("<DirEntry \'")
#         file = file.rstrip('\'>')
#         # print(file)
#         Data_Processing(file)
